# Back-translation of the 500 Pass 1 items (run order item 2)


In [2]:
# Cell 1. Install (about 2 minutes). transformers 4.46.3 is the version IndicTrans2 ran on in Panel 1.
!pip -q install "transformers==4.46.3" accelerate sentencepiece pandas IndicTransToolkit
print("installed. If Colab asks to restart the session, do it, then continue from cell 2.")

installed. If Colab asks to restart the session, do it, then continue from cell 2.


In [3]:
# Cell 2. Upload two files: 500_translated_gptoss.json and bangla_qe_pipeline.py
from google.colab import files
import os
up = files.upload()
for k in up:
    print(k, os.path.getsize(k), "bytes")
assert os.path.exists("bangla_qe_pipeline.py"), "bangla_qe_pipeline.py is missing: the notebook uses its nfc(), clean_bangla() and segment_labelled()"


Saving 500_translated_gptoss.json to 500_translated_gptoss (1).json
Saving bangla_qe_pipeline.py to bangla_qe_pipeline (1).py
500_translated_gptoss (1).json 2097647 bytes
bangla_qe_pipeline (1).py 20663 bytes


In [4]:
# Cell 3. Settings. Change nothing unless the run itself changes.
INPUT_JSON = "500_translated_gptoss.json"
RUN_NAME = "gptoss120b_pass1_500"
OUT_CSV = f"backtranslations_{RUN_NAME}.csv"

MODEL_NAME = "ai4bharat/indictrans2-indic-en-1B"
SRC_LANG, TGT_LANG = "ben_Beng", "eng_Latn"
NUM_BEAMS = 5
MAX_NEW_TOKENS = 256
LENGTH_PENALTY = 1.0
REPETITION_PENALTY = 1.0
SEED = 42
BATCH_SIZE = 16

USE_DRIVE = True          # True: the CSV lives in Google Drive and survives a disconnect. False: /content only.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/bangla_gap"
else:
    OUT_DIR = "/content"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = os.path.join(OUT_DIR, OUT_CSV)

import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("output file:", OUT_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
output file: /content/drive/MyDrive/bangla_gap/backtranslations_gptoss120b_pass1_500.csv


In [5]:
# Cell 4. HuggingFace login, then load IndicTrans2 (about 1 minute on T4).
# ai4bharat/indictrans2-indic-en-1B is a gated repo. Without a login the load fails with a
# "couldn't connect to huggingface.co" or a 401 error. Paste a read token from huggingface.co/settings/tokens.
from huggingface_hub import login
login(new_session=False)      # asks for the token once per runtime; a saved token is reused
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE).eval()
ip = IndicProcessor(inference=True)
print("loaded on", DEVICE)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

loaded on cuda


In [6]:
# Cell 5. Helpers. The Bangla goes through the same three steps the scoring pipeline uses:
# nfc() (the Panel 1 encoding fix), clean_bangla() (drops the ITEM_ID and QUESTION wrapper lines), segment_labelled() (stem and options).
import re
import bangla_qe_pipeline as p

def to_sentences(text):
    out = []
    for line in text.split("\n"):
        for s in re.split(r"(?<=[\u0964\?])\s*", line):
            s = s.strip()
            if not s:
                continue
            if len(s.split()) > 45:                                       # a very long run: split at full stops too
                out += [q.strip() for q in re.split(r"(?<=\.)\s+", s) if q.strip()]
            else:
                out.append(s)
    return out

def translate(sents):
    out = []
    for i in range(0, len(sents), BATCH_SIZE):
        chunk = sents[i:i + BATCH_SIZE]
        batch = ip.preprocess_batch(chunk, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
        enc = tok(batch, truncation=True, padding="longest", return_tensors="pt", return_attention_mask=True).to(DEVICE)
        with torch.inference_mode():
            gen = model.generate(**enc, num_beams=NUM_BEAMS, num_return_sequences=1, max_new_tokens=MAX_NEW_TOKENS,
                                 length_penalty=LENGTH_PENALTY, repetition_penalty=REPETITION_PENALTY, early_stopping=True)
        dec = tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        out += ip.postprocess_batch(dec, lang=TGT_LANG)
    return out

def backtranslate_text(bangla_raw):
    bn, _ = p.nfc(bangla_raw)
    bn = p.clean_bangla(bn)
    stem, opts = p.segment_labelled(bn)
    sents = to_sentences(stem)
    en_stem = translate(sents) if sents else []
    en_opts = translate([t for _, t in opts]) if opts else []
    tail = "\n".join(f"{lab}. {en}" for (lab, _), en in zip(opts, en_opts))       # the model's own labels, with a full stop like the source
    return (" ".join(en_stem) + "\n" + tail).strip()

import json
rec0 = json.load(open(INPUT_JSON, encoding="utf-8"))[0]
print(backtranslate_text(rec0["bangla"])[:800])

A 4670 g (10 lb 5 oz) male newborn was delivered full-term in a 26-year-old woman's womb after a long delivery. The Apgar score is 9 in both minutes 1 and 5. Tests in the delivery room revealed swelling, sensitivity, and crepitus in the left clavicle area. Movement of the left upper limb is reduced. Movement of the hands and wrists is normal. Swallowing reflexes are normal in both hands. An asymmetric Moro reflex is present. The rest of the test has no abnormalities, and enteroposterior X-rays confirm the diagnosis. Which of the following is the most appropriate next step in management?
A. Nerve conductivity test
B. Stabilization by Surgery
C. Physical therapy
D. Adding the pin sleeve to the shirt
E. hand splinting
F. MRI of the clavicle


In [7]:
# Cell 6. Back-translate all 500. Writes after every item and resumes if re-run.
import csv, json, time
import pandas as pd

recs = json.load(open(INPUT_JSON, encoding="utf-8"))
done = set()
if os.path.exists(OUT_PATH) and os.path.getsize(OUT_PATH) > 0:
    done = set(pd.read_csv(OUT_PATH, encoding="utf-8-sig")["item_id"].astype(str))
    print("resuming; already done:", len(done))
t_start = time.time()
n_new = 0
with open(OUT_PATH, "a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["item_id", "translator", "back_en", "seconds"])
    if not done:
        w.writeheader()
    for i, r in enumerate(recs, 1):
        if r["item_id"] in done or r.get("error") or not str(r.get("bangla") or "").strip():
            continue
        t0 = time.time()
        en = backtranslate_text(r["bangla"])
        w.writerow({"item_id": r["item_id"], "translator": r.get("model", "gpt-oss:120b"), "back_en": en,
                    "seconds": round(time.time() - t0, 2)})
        f.flush()
        n_new += 1
        if n_new % 25 == 0:
            print(f"{i} of {len(recs)}: {(time.time() - t_start) / 60:.1f} min so far")
print(f"finished: {n_new} new records in {(time.time() - t_start) / 60:.1f} min")

resuming; already done: 500
finished: 0 new records in 0.0 min


In [8]:
# Cell 7. Throughput, a look at one record, and download. Report the two numbers printed here.
df = pd.read_csv(OUT_PATH, encoding="utf-8-sig")
df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")            # BOM so Excel opens it correctly
sec = df["seconds"]
print(f"{len(df)} records. Total {sec.sum() / 60:.1f} min. {sec.mean():.1f} s per item. "
      f"Projection for 12,000 items: {sec.mean() * 12000 / 3600:.1f} h on a T4.")
print("empty back-translations:", int((df['back_en'].fillna('').str.strip() == '').sum()))
print("---", df['item_id'].iloc[0]); print(df['back_en'].iloc[0][:600])
files.download(OUT_PATH)

500 records. Total 13.2 min. 1.6 s per item. Projection for 12,000 items: 5.3 h on a T4.
empty back-translations: 0
--- US-00001
A 4670 g (10 lb 5 oz) male newborn was delivered full-term in a 26-year-old woman's womb after a long delivery. The Apgar score is 9 in both minutes 1 and 5. Tests in the delivery room revealed swelling, sensitivity, and crepitus in the left clavicle area. Movement of the left upper limb is reduced. Movement of the hands and wrists is normal. Swallowing reflexes are normal in both hands. An asymmetric Moro reflex is present. The rest of the test has no abnormalities, and enteroposterior X-rays confirm the diagnosis. Which of the following is the most appropriate next step in management?
A. Ner


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>